# Notebook for numerical results

## Needed things

In [2]:
import pandas as pd
import numpy as np
import os
from os import path
import matplotlib.pyplot as plt
import pickle

In [3]:
functions = ['ackley',
 'almost_perturbed_quadratic',
 'cube',
 'extended_feudenstein_and_roth',
 'extended_hiebert',
 'extended_himmelblau',
 'extended_psc1',
 'extended_rosenbrock',
 'extended_tridiagonal_1',
 'extended_tridiagonal_2',
 'extended_trigonometric',
 'extended_white_and_holst',
 'generalized_quartic',
 'generalized_rosenbrock',
 'generalized_white_and_holst',
 'griewank',
 'levy',
 'perturbed_quadratic',
 'perturbed_quadratic_diagonal',
 'power',
 'rastrigin',
 'schwefel',
 'sphere',
 'sum_of_different_powers',
 'trid',
 'zakharov']

In [ ]:
dims = [10] #, 100, 1000]
mus = [1e4, 1e5, 1e6, 1e7, 1e8]
taus = [1e-2, 1e-4, 1e-6, 1e-8, 1e-10]
algorithms = ['ASHGF', 'GD', 'SGES', 'ASGF', 'ASEBO']

In [5]:
def iterations(evs, dim, name):
    if name in ['ASGF', 'ASHGF']:
        return int(evs / (1 + dim * 4))
    else:
        return int(evs / (1 + dim * 2))
    
    
def evaluations(it, dim, name):

    if name in ['ASGF', 'ASHGF']:
        return it + it * dim * 4
    else:
        return it + it * dim * 2

## To execute only once

In [6]:
for dim in dims:
    main_folder = path.join('results', 'profiles', str(dim))
    mins = {}

    for function in functions:
        name = ''
        min_values = {}

        for mu in mus:
            min_value = 10**100
            min_algorithm = ''
            min_values[mu] = {}
            
            for algorithm in algorithms:
                loc = path.join(main_folder, function, algorithm, 'descent.csv')
                x = pd.read_csv(loc)

                min_x = float(x.iloc[:iterations(mu, dim, algorithm),:].min())
                min_values[mu][algorithm] = min_x

                if min_x < min_value:
                    min_value = min_x
                    min_algorithm = algorithm

                pd.DataFrame([[min_algorithm, min_value]], columns=['algorithm', 'minimum']).to_csv(path.join(main_folder, function, 'global_min_{}.csv'.format(mu)))

            mins[function] = min_values
    
    for function in functions:

        for mu in mus:
            convergence_test = []
            
            for algorithm in algorithms:
                loc = path.join(main_folder, function, algorithm, 'descent.csv')
                x = pd.read_csv(loc)
                x = x['descent'].to_list()

                loc_min_mu = loc = path.join(main_folder, function, 'global_min_{}.csv'.format(mu))
                min_mu = pd.read_csv(loc_min_mu)
                min_mu = float(min_mu['minimum'])

                it = min([iterations(mu, dim, algorithm), len(x)])
                temp = [algorithm]

                for tau in taus:
                    flag = False

                    for i in range(it):
                        if x[i] <= min_mu + tau * (x[0] - min_mu):
                            evs = evaluations(i, dim, algorithm)
                            temp.append(evs)
                            flag = True                            
                            break

                    if not flag:
                        temp.append(np.nan)

                convergence_test.append(temp)

            pd.DataFrame(convergence_test, columns=['algorithm', '1e-02', '1e-04', '1e-06', '1e-08', '1e-10']).to_csv(path.join(main_folder, function, 'evaluations_to_convergence_{}.csv'.format(mu)))

FileNotFoundError: [Errno 2] No such file or directory: 'results\\profiles\\10\\ackley\\ASHGF\\descent.csv'

In [ ]:
for dim in dims:
    main_folder = path.join('results', 'profiles', str(dim))
    
    for function in functions:

        for mu in mus:
            
            for algorithm in algorithms:
                loc = path.join(main_folder, function, 'evaluations_to_convergence_{}.csv'.format(mu))
                x = pd.read_csv(loc)

                for tau in ['1e-02', '1e-04', '1e-06', '1e-08', '1e-10']:
                    x[tau] = x[tau] / x[tau].min()

            pd.DataFrame(x, columns=['algorithm', '1e-02', '1e-04', '1e-06', '1e-08', '1e-10']).to_csv(path.join(main_folder, function, 'ratios_{}.csv'.format(mu)))

In [ ]:
for dim in dims:
    main_folder = path.join('results', 'profiles', str(dim))

    for function in functions:

        for mu in mus:

            for algorithm in algorithms:
                loc = path.join(main_folder, function, 'evaluations_to_convergence_{}.csv'.format(mu))
                x = pd.read_csv(loc)

                for tau in ['1e-02', '1e-04', '1e-06', '1e-08', '1e-10']:
                    x[tau] = x[tau] / (dim + 1)

            pd.DataFrame(x, columns=['algorithm', '1e-02', '1e-04', '1e-06', '1e-08', '1e-10']).to_csv(path.join(main_folder, function, 'ratios_data_{}.csv'.format(mu)))

In [ ]:
for dim in dims:
    main_folder = path.join('results', 'profiles', str(dim))
    num_problems = len(functions)
    performance_profiles = {}

    for mu in mus:
        performance_profiles[mu] = {}

        for tau in ['1e-02', '1e-04', '1e-06', '1e-08', '1e-10']:
            performance_profiles[mu][tau] = {}

            for algorithm in algorithms:
                temp = []

                for alpha in [i / 5 for i  in range(5, 16)]:
                    count = 0

                    for function in functions:
                        loc = path.join(main_folder, function, 'ratios_{}.csv'.format(mu))
                        x = pd.read_csv(loc)
                        data = x[x['algorithm'] == algorithm][tau]

                        if float(data) <= alpha:
                            count += 1

                    temp.append(count)

                performance_profiles[mu][tau][algorithm] = temp

    output = open(path.join(main_folder, 'performance_profiles_{}.pkl'.format(dim)), 'wb')
    pickle.dump(performance_profiles, output)
    output.close()

In [ ]:
for dim in dims:
    main_folder = path.join('results', 'profiles', str(dim))
    num_problems = len(functions)
    data_profiles = {}

    for mu in mus:
        data_profiles[mu] = {}

        for tau in ['1e-02', '1e-04', '1e-06', '1e-08', '1e-10']:
            data_profiles[mu][tau] = {}

            for algorithm in algorithms:
                temp = []

                for alpha in range(0, 1000, 50):
                    count = 0

                    for function in functions:
                        loc = path.join(main_folder, function, 'ratios_data_{}.csv'.format(mu))
                        x = pd.read_csv(loc)
                        data = x[x['algorithm'] == algorithm][tau]

                        if float(data) <= alpha:
                            count += 1

                    temp.append(count)

                data_profiles[mu][tau][algorithm] = temp

    output = open(path.join(main_folder, 'data_profiles_{}.pkl'.format(dim)), 'wb')
    pickle.dump(data_profiles, output)
    output.close()

## Things to actually run

### Performance profiles

In [ ]:
for dim in dims:
    main_folder = path.join('results', 'profiles', str(dim))
    num_problems = len(functions)

    pkl_file = open(path.join(main_folder, 'performance_profiles_{}.pkl'.format(dim)), 'rb')
    performance_profiles = pickle.load(pkl_file)
    pkl_file.close()

    for mu in mus:

        for tau in ['1e-02', '1e-04', '1e-06', '1e-08', '1e-10']:
            plt.rcParams.update({'font.size': 14})
            plt.figure(figsize=[6.4, 5.6])

            for algorithm in algorithms:
                if algorithm == 'GD':
                    plt.plot([i / 5 for i  in range(5, 16)], [a / num_problems for a in performance_profiles[mu][tau][algorithm]], label='ES')
                else:
                    plt.plot([i / 5 for i  in range(5, 16)], [a / num_problems for a in performance_profiles[mu][tau][algorithm]], label=algorithm)

            # plt.title(r'Perf. Profile - $Dim. = {}$, $\mu_L = {:.0e}$, $\tau = {}$'.format(dim, int(mu), tau))
            plt.title(r'$\tau = {}$'.format(tau))
            # plt.xscale('log')
            # plt.yscale('log')
            plt.xlabel(r'$\alpha$')
            plt.ylabel(r'$p_s (\alpha)$')
            plt.legend()
            plt.savefig(path.join('results', 'performance_profiles', str(dim), 'performance_profile_{}_{}_{}.png'.format(dim, int(mu), tau)), dpi=300)
            plt.show()

### Data profiles

In [ ]:
for dim in dims:
    main_folder = path.join('results', 'profiles', str(dim))
    num_problems = len(functions)

    pkl_file = open(path.join(main_folder, 'data_profiles_{}.pkl'.format(dim)), 'rb')
    data_profiles = pickle.load(pkl_file)
    pkl_file.close()

    for mu in mus:

        for tau in ['1e-02', '1e-04', '1e-06', '1e-08', '1e-10']:
            plt.figure(figsize=[6.4, 5.6])
            plt.rcParams.update({'font.size': 14})

            for algorithm in algorithms:
                if algorithm == 'GD':
                    plt.plot(range(0, 1000, 50), [a / num_problems for a in data_profiles[mu][tau][algorithm]], label='ES')
                else:
                    plt.plot(range(0, 1000, 50), [a / num_problems for a in data_profiles[mu][tau][algorithm]], label=algorithm)

            # plt.title(r'Data Profile - $Dim. = {}$, $\mu_L = {:.0e}$, $\tau = {}$'.format(dim, int(mu), tau))
            plt.title(r'$\tau = {}$'.format(tau))
            # plt.xscale('log')
            # plt.yscale('log')
            plt.xlabel(r'$\alpha$')
            plt.ylabel(r'$d_s (\alpha)$')
            plt.legend()
            plt.savefig(path.join(
                'results', 'data_profiles', str(dim),
                'data_profile_{}_{}_{}.png'.format(dim, int(mu), tau)),
                        dpi=300)
            plt.show()